# 📄 AI Research Paper Assistant

## The system allows users to:
-  Upload a research paper PDF
-  Generate a summary
-  Extract main contributions
-  Explain technical concepts
-  Ask questions about the paper

# Install Libraries

In [1]:
# Python packages
!pip install -qU \
    PyMuPDF \
    langchain \
    langchain-community \
    langchain-ollama \
    langchain-chroma \
    langchain-text-splitters \
    sentence-transformers \
    chromadb \
    pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 122.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.

In [2]:
# System dependency required by the Ollama installer
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 165 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (437 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [3]:
# Install Ollama (local LLM runtime)
!curl -fsSL https://ollama.com/install.sh | sh
!which ollama

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
/usr/local/bin/ollama


In [34]:
# Start the Ollama server in the background
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(10)
print("Ollama server started!")

Ollama server started!


In [5]:
# Pull the models used in this notebook:
# - qwen2.5:7b        -> chat/generation model
# - nomic-embed-text  -> embedding model for RAG
# !curl http://localhost:11434
!ollama pull qwen2.5:7b
!ollama pull nomic-embed-text
!ollama list



NAME                       ID              SIZE      MODIFIED               
nomic-embed-text:latest    0a109f422b47    274 MB    Less than a second ago    
qwen2.5:7b                 845dbda0ea48    4.7 GB    4 seconds ago             


In [6]:
!pip install faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 MB 7.2 MB/s eta 0:00:00


# Imports

In [7]:
# Standard library
import re
import time
import subprocess
import unicodedata

# PDF parsing
import fitz  # PyMuPDF

# LLM client
import ollama

# Text splitting / embeddings / vector store
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

/tmp/ipykernel_876/1884054741.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [8]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="missing ScriptRunContext! This warning can be ignored when running in bare mode.",
    category=RuntimeWarning
)

# PDF Processing Engineer

### Text Cleaning

In [9]:
# Extract Text
def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extract text from all pages of a PDF.
    """
    if not pdf_path.endswith('.pdf'):
        raise ValueError("Invalid file format. Only PDF files are supported.")

    document = fitz.open(pdf_path)
    text = ""

    for page in document:
        text += page.get_text()

    document.close()

    return text

In [10]:
# Clean Text
def clean_pdf_text(page_text: str) -> str:
    """
    Clean extracted PDF text for RAG applications.

    Steps:
    1. Normalize Unicode characters.
    2. Remove page break characters.
    3. Fix hyphenated words split across lines.
    4. Remove tabs.
    5. Remove isolated page numbers.
    6. Strip extra whitespace from each line.
    7. Collapse multiple spaces.
    8. Collapse excessive blank lines.
    """

    text = unicodedata.normalize("NFKC", page_text)
    text = text.replace("\x0c", "")
    text = text.replace("\t", " ")
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    lines = [line.strip() for line in text.split("\n")]
    lines = [line for line in lines if line]

    lines = [
        line for line in lines
        if not re.match(r'^(page\s*)?\d+\s*$', line, re.IGNORECASE)
    ]

    text = "\n".join(lines)
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [11]:
# Remove References
def remove_references(text: str) -> str:
    """
    Remove the References section and everything after it.
    """

    pattern = r"\nreferences\b.*"

    cleaned = re.sub(
        pattern,
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    return cleaned

### Chunking

In [12]:
# Chunking
def split_into_chunks(
    text: str,
    chunk_size: int = 1000,
    chunk_overlap: int = 200
):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunks = splitter.split_text(text)

    return chunks

### Run the PDF Pipeline

In [13]:
# Test PDF Processing
def process_pdf(pdf_path: str):
    raw_text = extract_text_from_pdf(pdf_path)
    cleaned_text = clean_pdf_text(raw_text)
    cleaned_text = remove_references(cleaned_text)
    return raw_text, cleaned_text

raw_text, clean_text = process_pdf("/content/rag.pdf")
chunks = split_into_chunks(clean_text)
print(f"Extracted {len(chunks)} chunks")

Extracted 42 chunks


# LLM Engineer

### Response Generation

In [14]:
# Create a reusable function to generate responses using the Qwen model

MODEL_NAME = "qwen2.5:7b"

def generate_response(
    prompt: str,
    temperature: float = 0.3,
    max_tokens: int = 1024
) -> str:
    """
    Generate a response using the Qwen model through Ollama.

    Args:
        prompt: The input prompt for the LLM.
        temperature: Controls randomness of the generated response.
        max_tokens: Maximum number of tokens to generate.

    Returns:
        The generated text response.
    """

    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": temperature,
            "num_predict": max_tokens
        }
    )

    return response["message"]["content"]

### Prompts

In [15]:
ROUTER_PROMPT = """
You are an intent classifier for a Research Paper Assistant. Classify the user's request into exactly ONE category.

Categories:
- SUMMARY: wants an overview/abstract of the paper (e.g. "Summarize this", "What is this paper about?")
- CONTRIBUTIONS: wants the paper's key contributions, novelty, or findings (e.g. "Main contributions", "What did they find?")
- CONCEPTS: wants a technical term or concept explained (e.g. "Explain Transformers", "What is attention?")
- QUESTION: wants a specific fact/detail from the paper (e.g. "What dataset was used?", "What accuracy did they get?")
- GREETING: small talk, greetings, or asking who you are (e.g. "Hi", "What can you do?")

If the request is ambiguous or fits more than one category, prefer QUESTION.
If the request is unrelated to the paper entirely, respond with QUESTION.

Respond with ONLY the category name in uppercase. No punctuation, no explanation, no extra words.

User Request:
{query}
"""

In [16]:
SUMMARY_PROMPT = """
You are an expert research assistant.

Read the research paper below carefully.

Rules:
- Use ONLY information from the paper.
- Do NOT invent missing information.
- Do NOT use outside knowledge.
- Do NOT include references or citations.
- Do NOT include related work.
- Do NOT include future work.
- If any section is not explicitly mentioned, write:
  Not explicitly mentioned in the paper.

Paper:
{paper}

---
IMPORTANT: Your response MUST follow EXACTLY this structure, with these exact headings,
and nothing else before or after it:

1. Research Objective
- Briefly describe the main problem the paper aims to solve.

2. Methodology
- Explain the proposed method or approach.
- Mention the model, algorithm, or architecture if applicable.

3. Main Findings
- Return 3-5 bullet points.
- Include only the paper's main experimental findings or achievements.

4. Conclusion
- Summarize the authors' final conclusion in one or two sentences.

Now write the summary using EXACTLY the structure above:
"""

In [17]:
CONTRIBUTIONS_PROMPT = """
You are an expert research analyst. Identify ONLY the original contributions this paper claims to make.

Rules:
- Include only what the authors present as new (their own method, result, dataset, or insight).
- Exclude background information, related work, prior methods, and future work.
- Exclude anything attributed to cited papers rather than this paper's own authors.
- If you are not confident something is an original contribution, do not include it.
- If no clear contributions are stated, respond exactly: No explicit contributions found in the paper.

Paper:
{paper}

---
IMPORTANT: Return your answer as a bulleted list, ordered from most to least significant.
For each contribution, write ONE clear sentence describing what was done and why it matters.
Return ONLY the bulleted list, nothing else before or after it.
"""

In [18]:
CONCEPTS_PROMPT = """
You are a patient teaching assistant helping a student understand a research paper.

Your task is to explain ONLY the exact concept explicitly mentioned in the user's question.

Instructions:
1. Identify the exact concept from the user's question.
2. Explain ONLY that concept.
3. Begin with a one-sentence definition of the concept.
4. Then explain how the concept is used in the research paper.
5. Use ONLY information found in the provided paper.
6. If the paper does not contain enough information to explain the requested concept, respond exactly with:

The requested concept is not discussed in the provided paper.

Rules:
- Never explain a different concept.
- Never summarize the entire paper.
- Never describe unrelated parts of the pipeline.
- Do not guess or use outside knowledge.
- Keep the explanation simple for undergraduate students.
- Use at most 3 concise bullet points.
- Avoid mathematical notation unless it is essential.

User Question:
{question}

Paper Context:
{paper}

---

Your response MUST use EXACTLY this format:

**Concept:** [Concept Name]

- definition.
- How the concept is used in this paper.
- One important role or purpose of the concept in this paper.
"""

In [19]:
QA_PROMPT = """
You are answering a question about a research paper using only the retrieved context below.

Rules:
- Answer using ONLY the context provided. Do not use outside knowledge, even if you know the answer.
- If the context only partially answers the question, answer what you can and note what's missing.
- If the answer is not present in the context at all, reply EXACTLY: I couldn't find this information in the uploaded paper.
- Keep the answer concise and directly address the question — don't restate the full context.
- Do not fabricate numbers, names, or details not present in the context.

Context:
{context}

Question:
{question}

Answer:
"""

In [20]:
GREETING_PROMPT = """
You are an AI assistant designed to greet users.
Respond with a friendly greeting and offer help.

Examples:
- Hello! How can I assist you today?
- Hi there! What can I do for you?
- Greetings! I'm here to help with your research paper.
"""

### Query Router

In [21]:
def classify_request(user_query: str):

    prompt = ROUTER_PROMPT.format(
        query=user_query
    )

    intent = generate_response(
        prompt,
        temperature=0.1,
        max_tokens=10
    )

    return intent.strip().upper()

# RAG Engineer

### Embeddings & Vector Store

In [22]:
# 4.1 Load Embedding Model
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)

print("Ollama Embedding model loaded successfully!")


Ollama Embedding model loaded successfully!


In [23]:
# 4.3 Create FAISS Index
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_texts(
    texts=chunks,
    embedding=embedding_model
)

print(f"FAISS Index created successfully with {vector_store.index.ntotal} chunks!")


FAISS Index created successfully with 42 chunks!


### Retrieval

In [24]:
def retrieve_context(
    query: str,
    k: int = 3
) -> str:
    """
    Retrieve the most relevant chunks from FAISS.
    """

    docs = vector_store.similarity_search(query, k=k)

    if not docs:
        return ""

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    return context

### Task's Functions

In [25]:
def generate_summary(
    paper_text: str
) -> str:

    prompt = SUMMARY_PROMPT.format(
        paper=paper_text
    )

    return generate_response(
        prompt,
        temperature=0.2,
        max_tokens=700
    )

In [26]:
def extract_contributions(
    paper_text: str
) -> str:

    prompt = CONTRIBUTIONS_PROMPT.format(
        paper=paper_text
    )

    return generate_response(
        prompt,
        temperature=0.2,
        max_tokens=500
    )

In [27]:
def explain_concepts(
    user_request: str
) -> str:

    context = retrieve_context(user_request)

    prompt = CONCEPTS_PROMPT.format(
        question=user_request,
        paper=context
    )

    return generate_response(
        prompt,
        temperature=0.2,
        max_tokens=700
    )


In [28]:
def answer_question(
    question: str
) -> str:

    context = retrieve_context(question)

    prompt = QA_PROMPT.format(
        context=context,
        question=question
    )

    return generate_response(
        prompt,
        temperature=0,
        max_tokens=300
    )

In [29]:
import random
def greeting(
    user_request: str
) -> str:

    dynamic_prompt = f"{GREETING_PROMPT}\n\nGenerate a unique greeting. Randomness factor: {random.random()}"

    return generate_response(
        dynamic_prompt,
        temperature=0.9,
        max_tokens=100
    )

### Routeing

In [30]:
def research_assistant(
    user_request: str,
    paper_text: str,
):
    """
    Main routing function.
    """

    intent = classify_request(user_request)

    print(f"\nDetected Intent: {intent}\n")

    if intent == "GREETING":
        return greeting(user_request)

    if intent == "SUMMARY":

        return generate_summary(
            paper_text
        )

    elif intent == "CONTRIBUTIONS":

        return extract_contributions(
            paper_text
        )

    elif intent == "CONCEPTS":

        return explain_concepts(
            user_request
        )

    elif intent == "QUESTION":

        return answer_question(
            user_request
        )

    else:

        return "Unable to classify the request."

# Test

In [35]:
query = 'can you summarize the pdf'
response = research_assistant(query, clean_text)
print(response)


Detected Intent: SUMMARY

1. Research Objective
- The paper aims to address the challenge of improving retrieval-augmented generation (RAG) systems by introducing a framework for cross-component prompt adaptation, specifically through GRADRAG. This approach seeks to enhance system outputs by allowing evaluation feedback to influence multiple upstream components rather than just refining the final answer.

2. Methodology
- The proposed method involves modeling RAG systems as computational graphs and using an evaluator agent to provide feedback across different stages of the pipeline: retrieval, evidence construction, and answer generation.
- GRADRAG employs a refinement cycle where each iteration updates prompts based on the previous stage's output, leading to coordinated improvements in system performance.

3. Main Findings
- Full GRADRAG variants outperform one-step baselines by 12–15 percentage points in pairwise comparisons across both flat and graph-based retrieval paradigms.
- Th

In [36]:
query = 'what is the main contributions?'
response = research_assistant(query, clean_text)
print(response)


Detected Intent: CONTRIBUTIONS

- GRADRAG introduced a framework for coordinating multiple agents in RAG pipelines through cross-component prompt adaptation, improving system outputs by 12–15 percentage points in pairwise comparisons.
- The evaluation showed that allowing evaluation feedback to influence upstream components rather than refining only the final answer generator consistently improves system performance.
- Results indicated that both flat and graph-based retrieval pipelines benefit from additional refinement under fixed budgets, supporting the use of GRADRAG for enhancing RAG systems.


In [37]:
query = 'how documents are segmented in chunk-based approach?'
response = research_assistant(query, clean_text)
print(response)


Detected Intent: QUESTION

Documents are segmented into overlapping text chunks using fixed character lengths (default 400 characters with a 40-character overlap).


In [38]:
query = 'What is FAISS?'
response = research_assistant(query, clean_text)
print(response)


Detected Intent: CONCEPTS

**Concept:** FAISS

- Definition: FAISS stands for "Facebook AI Similarity Search," a library designed to perform efficient similarity search among large sets of vectors. It supports various distance metrics and can be used with different indexing methods, making it suitable for tasks requiring fast approximate nearest neighbor searches.
- How the concept is used in this paper: In the research, FAISS is employed as one of the retrieval strategies during the iterative sub-query generation process. Specifically, after each iteration where context chunks are retrieved using BM25, the Retrieval Agent uses FAISS to perform dense similarity search among these chunks to find the most relevant ones.
- One important role or purpose of the concept in this paper: The use of FAISS helps improve the efficiency and effectiveness of the retrieval process by quickly identifying highly relevant text chunks that contribute to forming the final context for answer generation.


In [39]:
query = 'What is ROUGE evaluation metric?'
response = research_assistant(query, clean_text)
print(response)


Detected Intent: CONCEPTS

**Concept:** ROUGE evaluation metric

- Definition: ROUGE (Recall-Oriented Understudy for Gisting Evaluation) is a set of metrics designed to evaluate automatic summarization and machine translation systems by comparing generated summaries against reference summaries, focusing on n-gram overlap.
- How the concept is used in this paper: The paper mentions that traditional evaluation metrics like ROUGE correlate poorly with human judgments because they primarily measure lexical or embedding overlap rather than higher-level properties such as coherence, factuality, and reasoning. This indicates that while ROUGE was considered, it did not provide a comprehensive assessment of the quality of generated summaries.
- One important role or purpose of the concept in this paper: The discussion about ROUGE highlights the limitations of using such metrics for evaluating query-focused summarization, prompting the researchers to seek alternative evaluation methods that bet

In [40]:
query = 'how much llms achieves with human evaluators?'
response = research_assistant(query, clean_text)
print(response)


Detected Intent: QUESTION

The LLM achieves a 75.86% agreement rate with human evaluators according to the human evaluation results.


In [41]:
query = 'what is Gemini-2.5-Flash'
response = research_assistant(query, clean_text)
print(response)


Detected Intent: QUESTION

Gemini-2.5-Flash is mentioned as being used via the Google Generative Language API in the June 2025 version for tasks such as sub-query generation, flat retrieval, entity–relation extraction in graph retrieval, and community-level summarization and candidate-answer generation.


In [42]:
query = 'what is graph-based retrieval'
response = research_assistant(query, clean_text)
print(response)


Detected Intent: CONCEPTS

**Concept:** Graph-based retrieval

- Definition: A method for retrieving and processing information from documents by segmenting them into overlapping spans, extracting entities and relations to form a graph structure that is incrementally enriched across refinement iterations.
- How the concept is used in this paper: Documents are segmented into larger overlapping spans (default 1000 characters with a 200-character overlap) for reliable entity and relation extraction. A Graph Extraction Agent identifies these entities and relations, which are then aggregated into a graph structure that supports answer generation through refinement iterations.
- One important role or purpose of the concept in this paper: To construct and iteratively enrich an entity–relation graph from documents, providing structured information to the Answer Generation Agent for more accurate and contextually rich answers.


In [44]:
query = 'hello'
response = research_assistant(query, clean_text)
print(response)


Detected Intent: GREETING

Hello there! I hope you're having a great day. How can I assist you today? Whether you need information, help with a project, or just someone to chat with, feel free to let me know!
